# Phase 7 — Skill Gap Analysis

## Objective

This notebook implements the Skill Gap Analysis component of SkillSync AI.

The purpose of this phase is to identify the skills required by a Job Description that are not currently present in the candidate's resume.

The Skill Gap Analysis is built on top of the Resume ↔ Job Description Matching system developed in Phase 6.

## Input

The main input is:

- Candidate resume
- User-provided Job Description

The same skill extraction and normalization pipeline developed in earlier phases is used to extract skills from both inputs.

## Process

Resume
↓
Skill Extraction
↓
Resume Skills

Job Description
↓
Skill Extraction
↓
JD Required Skills

Resume Skills + JD Skills
↓
Resume ↔ JD Matching
↓
Matched Skills
Missing Skills
Additional Skills
Match Percentage
↓
Skill Gap Analysis

## Skill Gap Definition

A skill gap is a skill required by the Job Description that was not detected in the candidate's resume.

Therefore:

Skill Gap = Required JD Skills − Resume Skills

## Expected Output

The Skill Gap Analysis will provide:

1. Total required skills
2. Matched skills
3. Missing skills
4. Additional resume skills
5. Number of skill gaps
6. Skill match percentage
7. Categorized skill gaps

In [41]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from skill_extraction import (
    extract_and_classify_skills,
    match_resume_to_jd,
    build_extraction_vocabulary
)

print("Skill extraction module loaded successfully.")

Skill extraction module loaded successfully.


## Step 3 — Load the Skills Vocabulary

The cleaned skills dataset created during Phase 2 is used as the source vocabulary.

The project module provides `build_extraction_vocabulary()` to construct the final extraction vocabulary while preserving legitimate professional/domain skills.

This ensures that Phase 7 uses the same vocabulary-building logic as the rest of the SkillSync AI pipeline.

In [42]:
import pandas as pd

skills_df = pd.read_csv(
    "../data/processed/skills_clean.csv"
)

print("Skill vocabulary loaded.")
print("Number of source skills:", len(skills_df))

Skill vocabulary loaded.
Number of source skills: 36167


In [43]:
import re

skills_extraction = skills_df[
    ~skills_df["clean_skill"].str.fullmatch(r"\d+")
].copy()

skill_set = set(
    skills_extraction["clean_skill"]
    .dropna()
    .str.lower()
)

print("Extraction vocabulary size:", len(skill_set))

Extraction vocabulary size: 36149


In [44]:
generic_words = {
    "advanced",
    "analysis",
    "analytical",
    "codes",
    "collateral",
    "conflict",
    "customer",
    "data",
    "design",
    "features",
    "general",
    "government",
    "health",
    "hospital",
    "hotel",
    "insurance",
    "interpret",
    "layout",
    "management",
    "office",
    "operations",
    "paperwork",
    "participation",
    "plan",
    "policies",
    "prevention",
    "print",
    "public",
    "ratings",
    "records",
    "reporting",
    "reviews",
    "safety",
    "sales",
    "service",
    "skills",
    "statistics",
    "system",
    "teams",
    "terminology",
    "training",
    "transactions",
    "website",
    "organization",
    "professional",
    "responsibilities",
    "experience",
    "work",
    "working"
}

In [45]:
protected_skills = {
    "python",
    "java",
    "javascript",
    "c",
    "c++",
    "c#",
    "f#",
    "g++",
    "sql",
    "html",
    "css",
    "php",
    "ruby",
    "swift",
    "kotlin",
    "go",
    "r",
    "matlab",
    "scala",
    "perl",
    "bash",
    "powershell",
    "git",
    "github",
    "gitlab",
    "npm",
    "node.js",
    "react.js",
    "vue",
    "angular",
    "mongodb",
    "mysql",
    "postgresql",
    "oracle",
    "redis",
    "numpy",
    "pandas",
    "tensorflow",
    "pytorch",
    "keras",
    "tableau",
    "power bi",
    "docker",
    "kubernetes",
    "aws",
    "azure",
    "gcp",
    "linux",
    "android",
    "ios",
    "sap",
    "excel",
    "jira"
}

In [46]:
skill_set = build_extraction_vocabulary(
    skills_df,
    generic_words=generic_words,
    protected_skills=protected_skills
)

print("Final extraction vocabulary:", len(skill_set))

Final extraction vocabulary: 36107


## Step 5 — Controlled Job Description

A controlled Machine Learning Engineer Job Description is used to validate the Skill Gap Analysis.

Using the same Job Description as Phase 6 allows us to verify that Phase 7 correctly interprets the matching results rather than introducing a different extraction pipeline.

In [48]:
job_description = """
We are looking for a Machine Learning Engineer with experience in
Python, SQL, Machine Learning, Deep Learning, NumPy, Pandas,
Scikit-Learn, TensorFlow, PyTorch, Docker, Git, GitHub and AWS.

The candidate should have experience in data analysis, building
machine learning models, data preprocessing, model evaluation,
and software development.

Experience with PostgreSQL, MongoDB and Power BI is also preferred.
"""

In [49]:
matching_result = match_resume_to_jd(
    test_resume,
    job_description,
    skill_set,
    generic_jd_skills=generic_jd_skills
)

In [50]:
test_resume = """
Machine Learning Engineer with experience in Python, SQL, Machine Learning,
Deep Learning, NumPy, Pandas, Scikit-Learn, TensorFlow, PyTorch,
Docker, Git, GitHub, PostgreSQL and data analysis.

Experienced in developing machine learning models, preprocessing data,
evaluating models and working with software development tools.
"""

In [51]:
matching_result = match_resume_to_jd(
    test_resume,
    job_description,
    skill_set,
    generic_jd_skills=generic_jd_skills
)

print("Matching completed successfully.")

Matching completed successfully.


In [52]:
print("\n" + "=" * 60)
print("RESUME ↔ JD MATCHING RESULT")
print("=" * 60)

print("\nTotal Resume Skills:",
      matching_result["total_resume_skills"])

print("Total JD Skills:",
      matching_result["total_jd_skills"])

print("Matched Skills:",
      matching_result["matched_count"])

print("Missing Skills:",
      matching_result["missing_count"])

print("Additional Skills:",
      matching_result["additional_count"])

print("Skill Match Percentage:",
      matching_result["skill_match_percentage"], "%")


RESUME ↔ JD MATCHING RESULT

Total Resume Skills: 16
Total JD Skills: 18
Matched Skills: 14
Missing Skills: 4
Additional Skills: 2
Skill Match Percentage: 77.78 %


In [53]:
print("\nMatched Skills:")
for skill in matching_result["matched_skills"]:
    print("-", skill)

print("\nMissing Skills:")
for skill in matching_result["missing_skills"]:
    print("-", skill)

print("\nAdditional Resume Skills:")
for skill in matching_result["additional_skills"]:
    print("-", skill)


Matched Skills:
- data analysis
- deep learning
- docker
- git
- github
- machine learning
- numpy
- pandas
- postgresql
- python
- pytorch
- scikit-learn
- sql
- tensorflow

Missing Skills:
- aws
- mongodb
- power bi
- software development

Additional Resume Skills:
- models
- software development tools


## Step 9 — Skill Gap Analysis

The matching result from Phase 6 already contains the required information for Skill Gap Analysis.

The missing skills represent the candidate's identified skill gaps.

This phase converts the raw matching result into a structured Skill Gap Analysis result.

The analysis does not modify the matching percentage or invent additional skills. It organizes the information produced by the matching stage.

In [54]:
def analyze_skill_gap(matching_result):

    skill_gap_result = {
        "total_required_skills": matching_result["total_jd_skills"],
        "matched_skills": matching_result["matched_skills"],
        "missing_skills": matching_result["missing_skills"],
        "additional_skills": matching_result["additional_skills"],
        "matched_count": matching_result["matched_count"],
        "skill_gap_count": matching_result["missing_count"],
        "skill_match_percentage": matching_result["skill_match_percentage"]
    }

    return skill_gap_result

In [55]:
skill_gap_result = analyze_skill_gap(
    matching_result
)

print("Skill Gap Analysis created successfully.")

Skill Gap Analysis created successfully.


In [56]:
print("\n" + "=" * 60)
print("SKILL GAP ANALYSIS")
print("=" * 60)

print("\nTotal Required Skills:",
      skill_gap_result["total_required_skills"])

print("Matched Skills:",
      skill_gap_result["matched_count"])

print("Skill Gaps:",
      skill_gap_result["skill_gap_count"])

print("Skill Match Percentage:",
      skill_gap_result["skill_match_percentage"], "%")


SKILL GAP ANALYSIS

Total Required Skills: 18
Matched Skills: 14
Skill Gaps: 4
Skill Match Percentage: 77.78 %


In [57]:
print("\nMatched Skills:")

for skill in skill_gap_result["matched_skills"]:
    print("-", skill)

print("\nSkill Gaps:")

for skill in skill_gap_result["missing_skills"]:
    print("-", skill)

print("\nAdditional Resume Skills:")

for skill in skill_gap_result["additional_skills"]:
    print("-", skill)


Matched Skills:
- data analysis
- deep learning
- docker
- git
- github
- machine learning
- numpy
- pandas
- postgresql
- python
- pytorch
- scikit-learn
- sql
- tensorflow

Skill Gaps:
- aws
- mongodb
- power bi
- software development

Additional Resume Skills:
- models
- software development tools


In [58]:
def summarize_skill_gap(skill_gap_result):

    return {
        "required_skills": skill_gap_result["total_required_skills"],
        "matched_skills": skill_gap_result["matched_count"],
        "skill_gaps": skill_gap_result["skill_gap_count"],
        "additional_skills": len(
            skill_gap_result["additional_skills"]
        ),
        "match_percentage": skill_gap_result[
            "skill_match_percentage"
        ]
    }

In [59]:
skill_gap_summary = summarize_skill_gap(
    skill_gap_result
)

print("\n" + "=" * 60)
print("SKILL GAP SUMMARY")
print("=" * 60)

for key, value in skill_gap_summary.items():
    print(f"{key}: {value}")


SKILL GAP SUMMARY
required_skills: 18
matched_skills: 14
skill_gaps: 4
additional_skills: 2
match_percentage: 77.78


In [60]:
def interpret_match_percentage(match_percentage):

    if match_percentage >= 80:
        return "High skill overlap"

    elif match_percentage >= 60:
        return "Moderate skill overlap"

    elif match_percentage >= 40:
        return "Partial skill overlap"

    else:
        return "Low skill overlap"

In [61]:
interpretation = interpret_match_percentage(
    skill_gap_result["skill_match_percentage"]
)

print("\nSkill Match Interpretation:")
print(interpretation)


Skill Match Interpretation:
Moderate skill overlap


## Phase 7 Validation

The Skill Gap Analysis was validated using the controlled Machine Learning Engineer Job Description and candidate resume.

The process successfully:

- Extracted skills from the resume
- Extracted required skills from the Job Description
- Compared resume skills with required JD skills
- Identified matched skills
- Identified missing skills
- Identified additional resume skills
- Calculated the number of skill gaps
- Calculated the skill match percentage
- Generated a structured Skill Gap Analysis result

The Skill Gap Analysis is based on the normalized skill-overlap result produced by the Resume ↔ Job Description Matching component.

### Important Limitation

The current matching implementation primarily uses exact normalized skill matching.

Therefore, linguistically similar expressions may still be treated as different skills when they are not covered by the project's explicit alias rules.

For example, different wording or grammatical forms may require additional normalization or semantic NLP techniques.

This limitation can be addressed in a later enhancement using semantic similarity or embedding-based matching.

### Phase 7 Status

- [x] Load reusable skill extraction module
- [x] Build consistent extraction vocabulary
- [x] Extract resume skills
- [x] Extract Job Description skills
- [x] Perform Resume ↔ JD matching
- [x] Identify matched skills
- [x] Identify missing skills
- [x] Identify additional skills
- [x] Calculate number of skill gaps
- [x] Calculate skill match percentage
- [x] Generate structured Skill Gap Analysis
- [x] Validate controlled test case

**Phase 7 — Skill Gap Analysis: COMPLETE**

In [62]:
print("\n" + "=" * 60)
print("PHASE 7 — FINAL VALIDATION")
print("=" * 60)

print("\nTotal Required Skills:",
      skill_gap_result["total_required_skills"])

print("Matched Skills:",
      skill_gap_result["matched_count"])

print("Skill Gaps:",
      skill_gap_result["skill_gap_count"])

print("Additional Resume Skills:",
      len(skill_gap_result["additional_skills"]))

print("Skill Match Percentage:",
      skill_gap_result["skill_match_percentage"], "%")

print("\n--- MATCHED SKILLS ---")
for skill in skill_gap_result["matched_skills"]:
    print("-", skill)

print("\n--- SKILL GAPS ---")
for skill in skill_gap_result["missing_skills"]:
    print("-", skill)

print("\n--- ADDITIONAL RESUME SKILLS ---")
for skill in skill_gap_result["additional_skills"]:
    print("-", skill)

print("\n--- INTERPRETATION ---")
print(
    interpret_match_percentage(
        skill_gap_result["skill_match_percentage"]
    )
)


PHASE 7 — FINAL VALIDATION

Total Required Skills: 18
Matched Skills: 14
Skill Gaps: 4
Additional Resume Skills: 2
Skill Match Percentage: 77.78 %

--- MATCHED SKILLS ---
- data analysis
- deep learning
- docker
- git
- github
- machine learning
- numpy
- pandas
- postgresql
- python
- pytorch
- scikit-learn
- sql
- tensorflow

--- SKILL GAPS ---
- aws
- mongodb
- power bi
- software development

--- ADDITIONAL RESUME SKILLS ---
- models
- software development tools

--- INTERPRETATION ---
Moderate skill overlap
